# Reproducing AWARE2.0
This notebook allows to trace the calculation process of AWARE2.0. AWARE2.0 is an update of the Available Water Remaining (AWARE) method, which was originally published in Boulay et al. (2018). AWARE2.0 is documented in Seitfudem et al. (2025).

- Boulay, A.-M., Bare, J., Benini, L., Berger, M., Lathuillière, M. J., Manzardo, A., Margni, M., Motoshita, M., Núñez, M., Pastor, A. V., Ridoutt, B., Oki, T., Worbe, S., & Pfister, S. (2018). The WULCA consensus characterization model for water scarcity footprints: Assessing impacts of water consumption based on available water remaining (AWARE). International Journal of Life Cycle Assessment, 23(2), 368–378. https://doi.org/10.1007/s11367-017-1333-8

- Seitfudem, G., Berger, M., Schmied, H. M., & Boulay, A.-M. (2025). The updated and improved method for water scarcity impact assessment in LCA, AWARE2.0. Journal of Industrial Ecology, 29(3), 891–907. https://doi.org/10.1111/jiec.70023


> *Note*
>
> Since this code will load large amounts of data into your memory, it is recommended to have at least 16 GB RAM available, better 32.
> It is possible to rewrite it for being less memory-heavy, e.g. by moving from pickle to a lazy-loaded file format, but due to time constraints this has not been done yet.

## Generate AWARE_DB
To preprocess and "import" the raw WaterGAP data, an AWARE_data object is created. The data from WaterGAP used as input has to be donwloaded into the Input folder. For the URLs to the files, see Table S1 of Supplementary Information 2 of Seitfudem et al. (2025). ( https://onlinelibrary.wiley.com/action/downloadSupplement?doi=10.1111%2Fjiec.70023&file=jiec70023-sup-0002-SuppMat.pdf )

In [1]:
import AWARE_data as AD
import pandas as pd
import AWARE_data_import as AI
import AWARE_CF_equation as AE
import pickle
from io import BytesIO
from urllib.request import urlopen
import ssl
import certifi

inputpath = "Input/"
outputpath = "Output/"

We load the input data and convert it into an "AWARE_data" object (takes around 15 to 20 minutes):

In [ ]:
# create an AWARE_Data object. This object will be pickled and hold all the data required for AWARE2.0 (and more)
gcm = "gswp3-w5e5"
ghm = "WG22e"
AWARE_DB = AD.AWARE_data("AWARE2-0 reproduced", GHM=ghm, GCM=gcm, scenario="hist")

dispath = inputpath + "watergap2-2e_gswp3-w5e5_obsclim_histsoc_default_dis_global_monthly_1901_2019.nc"
runoffpath = inputpath + "watergap2-2e_gswp3-w5e5_obsclim_histsoc_nowatermgt_qtot_global_monthly_1901_2019.nc"
conspath = dispath.replace("dis","atotuse")
natdispath = dispath.replace("default","nowatermgt")
mapping = inputpath + "mapping/"

InlandSinkInflowCells = pd.read_csv(mapping + "InlandSinkInflowCells.csv", index_col=["lat","lon"])
AWARE_DB.addAuxiliary(InlandSinkInflows= InlandSinkInflowCells)

InlandSinkBasins = pd.read_csv(mapping + "InlandSinkBasins.csv", index_col=["Basin_ID"])
AWARE_DB.addAuxiliary(InlandSinkBasins= InlandSinkBasins)

Deltas = pd.read_csv(mapping + "Deltas.csv", index_col=["Basin_ID"])
AWARE_DB.addAuxiliary(DeltaBasins= Deltas)

BasinArea = pd.read_csv(mapping + "Area.csv", index_col=["Basin_ID"])
AWARE_DB.addAuxiliary(Area = BasinArea)

BasinGrid=pd.read_csv(mapping + "BasinGrid.csv", index_col=["lat","lon"])
BasinGrid.sort_index(ascending=False, inplace=True)

yearwise_dis = AI.return_dataframes_given_dis_path(dispath, BasinGrid, startyear=1960)

yearwise_dis_nat = AI.return_dataframes_given_dis_path(natdispath, BasinGrid, startyear=1960)

yearwise_runoff_nat = AI.return_dataframes_given_cons_path(runoffpath,
                                                     "qtot",
                                                    BasinArea,
                                                    BasinGrid,
                                                    "runoff",
                                                    startyear=1960)
yearwise_consumption = AI.return_dataframes_given_cons_path(conspath,
                                                     "atotuse",
                                                    BasinArea,
                                                    BasinGrid,
                                                    "cons",
                                                    startyear=1960)

AWARE_DB.addDischarge("dis_MAINTABLE", AD.AWARE_data_entry(yearwise_dis["MAINTABLE"], variable="dis_MAINTABLE",
                                            times="yearspecific", unit = "m³/month", aux=yearwise_dis["gridcells_auxiliary"]))
AWARE_DB.addDischarge("dis", AD.AWARE_data_entry(yearwise_dis["Basin"], variable="dis",
                            times="yearspecific", unit = "m³/month"))

AWARE_DB.addDischarge("disnat_MAINTABLE", AD.AWARE_data_entry(yearwise_dis_nat["MAINTABLE"], variable="disnat_MAINTABLE",
                                            times="yearspecific", unit = "m³/month", aux=yearwise_dis_nat["gridcells_auxiliary"]))
AWARE_DB.addDischarge("disnat", AD.AWARE_data_entry(yearwise_dis_nat["Basin"], variable="dis",
                            times="yearspecific", unit = "m³/month"))

AWARE_DB.addRunoff("qtotnat_MAINTABLE", AD.AWARE_data_entry(yearwise_runoff_nat["MAINTABLE"], variable="qtotnat_MAINTABLE",
                                            times="yearspecific", unit = "mixed (m³ and the raw netCDF value, usually kg/m²s)", aux=yearwise_runoff_nat["gridcells_auxiliary"]))
AWARE_DB.addRunoff("qtotnat", AD.AWARE_data_entry(yearwise_runoff_nat["Basin"],variable="qtotnat",
                            times="yearspecific", unit = "m³/month or m³/year"))

AWARE_DB.addConsumption("atotuse_MAINTABLE", AD.AWARE_data_entry(yearwise_consumption["MAINTABLE"], variable="atotuse_MAINTABLE",
                                            times="yearspecific", unit = "mixed (m³ and the raw netCDF value, usually kg/m²s)", aux=yearwise_consumption["gridcells_auxiliary"]))
AWARE_DB.addConsumption("atotuse", AD.AWARE_data_entry(yearwise_consumption["Basin"], variable="atotuse",
                            times="yearspecific", unit = "m³/month or m³/year"))


# not required for basin-level CFs but eventually for the spatiotemporal aggregations:
consumption_sectors = ["pirruse","pliveuse","pmanuse","pelecuse","pdomuse"]
for use in consumption_sectors:
    potential_cons = AI.return_dataframes_given_cons_path(conspath.replace("atotuse",use),
                                                        use,
                                                        BasinArea,
                                                        BasinGrid,
                                                        "cons",
                                                        startyear=1960)
    AWARE_DB.addConsumption(use+"_MAINTABLE", AD.AWARE_data_entry(potential_cons["MAINTABLE"], variable=use+"_MAINTABLE",
                                            times="yearspecific", unit = "mixed (m³ and the raw netCDF value, usually kg/m²s)", aux=potential_cons["gridcells_auxiliary"]))
    AWARE_DB.addConsumption(use, AD.AWARE_data_entry(potential_cons["Basin"], variable=use,
                            times="yearspecific", unit = "m³/month or m³/year"))

#calculate total potential water consumption 
AWARE_DB.AddUpVariables_consumption({"ptotuse": consumption_sectors})
AWARE_DB.AddUpVariables_consumption({"ptotuse_MAINTABLE":
                                     [f"{x}_MAINTABLE" for x in consumption_sectors]})

AWARE_DB.save(outputpath+'AWARE_DB_'+"_".join(["GHM",ghm,"GCM",gcm,"SCE","hist"]),
              dump_seperately=True, entire_DB=False, remove_from_main_DB=True)

created AWARE data object AWARE2-0 reproduced  for GHM: WG22e , GCM: gswp3-w5e5 scenario: hist
kg m-2 s-1
kg m-2 s-1
added discharge object dis_MAINTABLE, yearspecific, m³/month
added discharge object dis, yearspecific, m³/month
added discharge object disnat_MAINTABLE, yearspecific, m³/month
added discharge object dis, yearspecific, m³/month
added runoff object qtotnat_MAINTABLE, yearspecific, mixed (m³ and the raw netCDF value, usually kg/m²s)
added runoff object qtotnat, yearspecific, m³/month or m³/year
added consumption object atotuse_MAINTABLE, yearspecific, mixed (m³ and the raw netCDF value, usually kg/m²s)
added consumption object atotuse, yearspecific, m³/month or m³/year
kg m-2 s-1
added consumption object pirruse_MAINTABLE, yearspecific, mixed (m³ and the raw netCDF value, usually kg/m²s)
added consumption object pirruse, yearspecific, m³/month or m³/year
kg m-2 s-1
added consumption object pliveuse_MAINTABLE, yearspecific, mixed (m³ and the raw netCDF value, usually kg/m²s)

We now have formatted and saved our input data for further analysis. To load the input data:

In [3]:
with open(outputpath+'AWARE_DB_'+"_".join(["GHM",ghm,"GCM",gcm,"SCE","hist"]), 'rb') as file:
    AWARE_DB = pickle.load(file)
AWARE_DB.load_externally_saved()
AWARE_DB

loading cons loading dis loading runoff loading EWR loading EFR loading external consumption external_cons has empty dictionary


AWARE Database
GHM WG22e 
GCM: gswp3-w5e5 
scenario: hist 
descriptor: AWARE2-0 reproduced
----------
content:

['atotuse_MAINTABLE', 'atotuse', 'pirruse_MAINTABLE', 'pirruse', 'pliveuse_MAINTABLE', 'pliveuse', 'pmanuse_MAINTABLE', 'pmanuse', 'pelecuse_MAINTABLE', 'pelecuse', 'pdomuse_MAINTABLE', 'pdomuse', 'ptotuse', 'ptotuse_MAINTABLE']
['dis_MAINTABLE', 'dis', 'disnat_MAINTABLE', 'disnat']
['qtotnat_MAINTABLE', 'qtotnat']

In [4]:
AWARE_DB.ALWAYS_FORCE_UPDATES = True

In [5]:
# calculate inland sink values
AWARE_DB.InlandSinkAvailabilities()
# define longterm average periods
periods = {x:[1960+10*x,1989+10*x] for x in range(4)}
periods["EWR_period"] = [1960,2010]
# calculate longterm averages
AWARE_DB.LongtermAveraging(periods=periods, inlandsinks="approximations")
#calculate EFRs and EWRs
AWARE_DB.calculateEFRs()
AWARE_DB.calculateEWRs()
#do delta merging
AWARE_DB.DoDeltaMerging()

no qtot available, so no qtot at outflow sink calculated (does not affect inland sink calculations)
added discharge object discharge_inflow_to_InlandSink, annual, m³/month
added discharge object discharge_inflow_to_InlandSink_separated_gridcells, annual, m³/month
added discharge object naturalized_discharge_inflow_to_InlandSink, annual, m³/month
added discharge object naturalized_discharge_inflow_to_InlandSink_separated_gridcells, annual, m³/month
added runoff object naturalized_runoff_at_outflow_grid_cells, annual, m³/month
added consumption object atotuse_at_outflow_grid_cells, annual, m³/month
added discharge object approximated_available_water_after_human_consumption_in_inlandsinks, annual, m³/month
added discharge object approximated_naturally_available_water_in_inlandsinks, annual, m³/month
added discharge object discharge_at_outflows_including_approximated_inland_sinks, annual, m³/month
added discharge object naturalized_discharge_at_outflows_including_approximated_inland_sinks,

At this point, the AWARE CFs can already be calculated. However, since the source data from WaterGAP does not update water demands after a certain point in time (depends on water use sector), for AWARE2.0 there was a postprocessing step applied (see supplementary material linked above). We now apply the relative changes determined in that postprocessing step to the relevant variables.


We need:

- dis_name = "dis_with_inland_sinks_LT_adj_w_cons_change_w_deltas"
- EWR_name = "EWR_deltas_merged"
- HWC_name = "adjusted_atotuse_basin"

In [6]:
# relative change of atotuse per basin
postproc_path = "Input/postprocessing/"
adjusted_atotuse = dict()
for year in range(2009,2020):
    diff_atotuse = pd.read_csv(f"{postproc_path}/atotuse_postproc_diff_{year}.csv", index_col="Basin_ID")
    adjusted_atotuse[year] = AWARE_DB.consumption["atotuse"].data[year][diff_atotuse.columns]+diff_atotuse
AWARE_DB.addConsumption("adjusted_atotuse_basin", AD.AWARE_data_entry(adjusted_atotuse, variable="atotuse_adjusted_in_postprocessing",
                            times="yearspecific", unit = "m³/month"))

# relative change in discharge per basin
adjusted_discharge = dict()
for year in range(2009,2020):
    diff_dis = pd.read_csv(f"{postproc_path}/dis_with_inland_sinks_postproc_diff_{year}.csv", index_col="Basin_ID")
    adjusted_discharge[year] = AWARE_DB.discharge["dis_with_inland_sinks"].data[year]+diff_dis
AWARE_DB.addDischarge("simulated_adjusted_ActAvail_basin", AD.AWARE_data_entry(adjusted_discharge, variable="Adjusted_Discharge_Basin_from_resimulation__taken_from_outflow_cell_after_adjusting",
                            times="yearspecific", unit = "m³/month"))



#### ONLY REQUIRED FOR SPATIOTEMPORAL AGGREGATIONS
# relative change of potential water consumption per grid cell
adjusted_gridcell_ptotuse = dict()
for year in range(2009,2020):
    diff = pd.read_csv(f"{postproc_path}/ptotuse_extrap_cells_postproc_diff_{year}.csv").set_index(["lat","lon"])
    orig = AWARE_DB.consumption["ptotuse_MAINTABLE"].data[year][[f"{m}_Area_x_Cons_m3" for m in AWARE_DB.Mon_List]].sort_index()
    orig.columns = AWARE_DB.Mon_List
    adjusted_gridcell_ptotuse[year] = orig + diff
AWARE_DB.addConsumption("ptotuse_extrap_cells", AD.AWARE_data_entry(adjusted_gridcell_ptotuse, variable="pot_consumption_all_sectors_gridcells_extrapolated_and_scaled_for_2009_to_2019",
                            times="yearspecific", unit = "m³/month"))

# relative change of potential water consumption (excluding irrigation) per grid cell
adjusted_gridcell_ptotuse = dict()
for year in range(2009,2020):
    diff = pd.read_csv(f"{postproc_path}/ptotuse_extrap_cells_no_irrig_postproc_diff_{year}.csv").set_index(["lat","lon"])
    orig = AWARE_DB.consumption["ptotuse_MAINTABLE"].data[year][[f"{m}_Area_x_Cons_m3" for m in AWARE_DB.Mon_List]].sort_index()
    orig.columns = AWARE_DB.Mon_List
    adjusted_gridcell_ptotuse[year] = orig + diff
AWARE_DB.addConsumption("ptotuse_extrap_cells_no_irrig", AD.AWARE_data_entry(adjusted_gridcell_ptotuse, variable="pot_consumption_without_irrigation_gridcells_linearly_extrapolated_for_2009_to_2019",
                            times="yearspecific", unit = "m³/month"))

# get data for irrigation only per grid cell
adjusted_gridcell_ptotuse = dict()
for year in range(2009,2020):
    total = AWARE_DB.consumption["ptotuse_extrap_cells"].data[year]
    nonirri = AWARE_DB.consumption["ptotuse_extrap_cells_no_irrig"].data[year]
    adjusted_gridcell_ptotuse[year] = total - nonirri
AWARE_DB.addConsumption("irri_extrap_cells", AD.AWARE_data_entry(adjusted_gridcell_ptotuse, variable="area-scaled_potential_irrigation_water_consumption",
                            times="yearspecific", unit = "m³/month"))

added consumption object atotuse_adjusted_in_postprocessing, yearspecific, m³/month
added discharge object Adjusted_Discharge_Basin_from_resimulation__taken_from_outflow_cell_after_adjusting, yearspecific, m³/month
added consumption object pot_consumption_all_sectors_gridcells_extrapolated_and_scaled_for_2009_to_2019, yearspecific, m³/month
added consumption object pot_consumption_without_irrigation_gridcells_linearly_extrapolated_for_2009_to_2019, yearspecific, m³/month
added consumption object area-scaled_potential_irrigation_water_consumption, yearspecific, m³/month


In [7]:
AWARE_DB.recalculateLongtermDischargeValues(discharge_type = "dis_with_inland_sinks") 
AWARE_DB.calculateLongtermActualConsumption() # not necessary for CFs but nevertheless interesting to have the longterm average actual consumption

added discharge object dis_with_inland_sinks_LT_average_average_mode_MEAN___adjusted_with_consumption_change, longterm, m³/month
added discharge object dis_with_inland_sinks_LT_average_average_mode_MEAN___adjusted_with_consumption_change_with_deltas, longterm, m³/month
succesfully merged deltas for dis_with_inland_sinks_LT_adj_w_cons_change
added consumption object atotuse___adjusted_to_extrapolations_and_availability, longterm, m³/month


## Recalculate AWARE CFs from preprocessed input data

create a new object that contains the standard AWARE variables:
- ActAvail
- EWR
- area
- HWC_inventory (longterm or snapshot)

In [8]:
# for Standard_AWARE20:
dis_name = "dis_with_inland_sinks_LT_adj_w_cons_change_w_deltas"
EWR_name = "EWR_deltas_merged"
HWC_name = "adjusted_atotuse_basin"
HWC_year = 2019
subbasin_info_path = inputpath+"mapping/SubBasinInformation.xlsx"
AMDwa_period = 3

AWARECF_equation = AE.AWARE_CF_equation(ActAvail = AWARE_DB.discharge[dis_name],
                                            EWR = AWARE_DB.EWR[EWR_name],
                                            area = AWARE_DB.AreaDeltaAllocated,
                                            HWC = AWARE_DB.consumption[HWC_name].data[HWC_year],
                                            subbasin_info_path = subbasin_info_path,
                                            AMDwa_period = AMDwa_period
                                            )
AWARECF_equation.calculate_CFs()
print(AWARECF_equation.AMDwa)
AWARECF_equation.save(rf"{outputpath}AWARE_equation")

created AWARE_equation object
doing basin subdivision
start iteration 0
870 instances of AMD too high compared to local availability
Basins affected out of 129 : 86
start iteration 1
146 instances of AMD too high compared to local availability
Basins affected out of 129 : 86
start iteration 2
0 instances of AMD too high compared to local availability
start iteration 3
0 instances of AMD too high compared to local availability
start iteration 4
0 instances of AMD too high compared to local availability
start iteration 5
0 instances of AMD too high compared to local availability
start iteration 6
0 instances of AMD too high compared to local availability
start iteration 7
0 instances of AMD too high compared to local availability
doing basin subdivision
start iteration 0
878 instances of AMD too high compared to local availability
Basins affected out of 129 : 86
start iteration 1
160 instances of AMD too high compared to local availability
Basins affected out of 129 : 86
start iteration 

The AWARE2.0 CFs in native resolution are here:

In [9]:
url = "https://zenodo.org/records/16332127/files/AWARE20_Native_CFs.xlsx?download=1"
ssl_context = ssl.create_default_context(cafile=certifi.where())
with urlopen(url, context=ssl_context) as response:
    native_CFs = pd.read_excel(BytesIO(response.read()), sheet_name="native_CFs")

native_CFs = native_CFs.set_index("Basin_ID")

# compare CFs by using relative difference
months = AWARE_DB.Mon_List

print("relative difference between this notebook and the CFs on Zenodo:")
diagnosis = (AWARECF_equation.CFs[3][months]-native_CFs[months]).div(native_CFs[months]).stack().describe().loc[["min","max","mean"]]
display(diagnosis)

relative difference between this notebook and the CFs on Zenodo:


min    -0.004925
max     0.004985
mean    0.000009
dtype: float64

Note that the Zenodo Excel values are only provided with 3 significant figures, since the overall uncertainty of the CFs is large. So, with ±0.5%, this is a very close approximation!

...but: We are missing Antipodes island, because the coordinates have changed between different versions of the WaterGAP files (see: https://www.isimip.org/gettingstarted/input-data-bias-adjustment/details/41/#:~:text=Caveats). If required, this could be mitigated by changing the corresponding coordinate in the CSV file BasinGrid.csv.

In [10]:
# missing basin (Antipodes Island)
set(native_CFs.index)-set(AWARECF_equation.CFs[3].index)

{67311}

## weighting to annual basin CFs

In [11]:
weighting = AWARE_DB.consumption["ptotuse_extrap_cells"].data[2019].join(BasinGrid[["Basin_ID"]]).groupby("Basin_ID").sum()
annual_CFs_unspecified = AWARECF_equation.get_annual_weighted_CFs(AWARECF_equation.CFs[3], AWARE_DB.Mon_List, weighting)

weighting = AWARE_DB.consumption["irri_extrap_cells"].data[2019].join(BasinGrid[["Basin_ID"]]).groupby("Basin_ID").sum()
annual_CFs_irrigation = AWARECF_equation.get_annual_weighted_CFs(AWARECF_equation.CFs[3], AWARE_DB.Mon_List, weighting)

In [12]:
print("Irrigation CF: relative difference between this notebook and the CFs on Zenodo:")
diagnosis = (annual_CFs_irrigation-native_CFs["annual_agri"]).div(native_CFs["annual_agri"].loc[native_CFs["annual_agri_is_arithm_mean"] == False]).describe().loc[["min","max","mean"]]
display(diagnosis)

print("Unspecified CF: relative difference between this notebook and the CFs on Zenodo:")
diagnosis = (annual_CFs_unspecified-native_CFs["annual_unspecified"]).div(native_CFs["annual_unspecified"].loc[native_CFs["annual_unspecified_is_arithm_mean"] == False]).describe().loc[["min","max","mean"]]
display(diagnosis)

Irrigation CF: relative difference between this notebook and the CFs on Zenodo:


min    -0.004941
max     0.004876
mean   -0.000003
dtype: float64

Unspecified CF: relative difference between this notebook and the CFs on Zenodo:


min    -0.004668
max     0.004606
mean   -0.000002
dtype: float64

In [13]:
# save the database for next time
AWARE_DB.save(AWARE_DB.Path,
              dump_seperately=True, entire_DB=False, remove_from_main_DB=True)

saving...cons saving...dis saving...runoff saving...EWR saving...EFR saving...external_cons removing from main DB
saving... AWARE_DB_GHM_WG22e_GCM_gswp3-w5e5_SCE_hist
done
done
